# SLAM
**SLAM (Simultaneous Localization And Mapping)** estimates the trajectory of a moving sensor (localization) while building a map of the environment (mapping) at the same time. With a depth sensor or a LiDAR, every observation is a point cloud *frame*, and SLAM has to figure out where each frame was taken and how the frames fit together into one map.

A point cloud SLAM is usually divided into two parts:
- **Front-end (odometry)**: estimate the relative motion between consecutive frames by *registration* (point cloud alignment).
- **Back-end**: refine the whole trajectory with pose-graph optimization and *loop closure* to reduce the drift that accumulates over time.

This section implements a minimal front-end SLAM: we estimate the sensor trajectory by chaining frame-to-frame [ICP registration](registration.ipynb), and merge the frames into a global map using the estimated poses.

**Note**: to keep the example simple and reproducible, we create a synthetic frame sequence without occlusion or sensor noise, and we do not perform loop closure. A real SLAM has to handle all of these.

In [ ]:
%load_ext autoreload
%autoreload 2

This section uses the following code:

In [ ]:
import numpy as np

# for SLAM
from tutlibs.slam import sequential_registration, build_map, trajectory

# for creating a synthetic frame sequence
from tutlibs.transformation import TransformationMatrix as tm
from tutlibs.sampling import random_sampling

# for description
from tutlibs.io import Points as io
from tutlibs.utils import single_color
from tutlibs.visualization import JupyterVisualizer as jv
import inspect

## Create a synthetic frame sequence
We use a single point cloud as the environment (the *scene*) and simulate a sensor that moves along a trajectory. The sensor pose $M_i$ is a frame-to-world transform. The frame observed at time $i$ is the scene expressed in the sensor coordinate, i.e. $S_i = M_i^{-1} P$ where $P$ is the scene.

Here the sensor moves forward while slowly turning.

In [ ]:
# load a point cloud and use it as the environment (scene).
scene, _, _ = io.read("../data/bunny_pc.ply")
scene = scene[random_sampling(scene, 2000)]  # subsample for faster ICP

# ground-truth sensor poses along a forward + turning trajectory.
num_frames = 8
gt_poses = []
frames = []
for i in range(num_frames):
    angle = np.deg2rad(6 * i)  # yaw angle (rotation around y axis)
    rotation = np.array(
        [
            [np.cos(angle), 0, np.sin(angle)],
            [0, 1, 0],
            [-np.sin(angle), 0, np.cos(angle)],
        ]
    )
    translation = np.array([0.02 * i, 0.0, 0.0])
    pose = tm.composite(
        [tm.from_translation(translation), tm.from_rotation(rotation)]
    )  # sensor-to-world transform
    gt_poses.append(pose)
    # frame = scene seen from the sensor = world points in sensor coordinate.
    frame = tm.transformation(scene, np.linalg.inv(pose))
    frames.append(frame)

print("number of frames:", len(frames))

# visualize the frames in their own (sensor) coordinate.
# because each frame is taken from a different pose, they do not overlap.
colors = [
    "#ff0000", "#ff8800", "#cccc00", "#00cc00",
    "#00cccc", "#0000ff", "#8800ff", "#ff00ff",
]
objs = [
    jv.point(frame, single_color(colors[i % len(colors)], len(frame)))
    for i, frame in enumerate(frames)
]
jv.display(objs)

The frames above are misaligned because each one is observed from a different sensor pose. SLAM does not know the poses; it has to recover them from the frames only.

## Front-end: estimate the trajectory by sequential registration
For every pair of consecutive frames we estimate the relative motion with ICP, and accumulate the relative motions to obtain the global pose of each frame. The first frame defines the origin of the trajectory.

In [ ]:
print(inspect.getsource(sequential_registration))

The `sequential_registration` process is as follows:
1. Set the first frame as the origin (identity pose).
2. For each frame $i$, run ICP so that frame $i$ is aligned onto frame $i-1$. ICP returns the relative transform $T_i$ (the motion between the two frames).
3. Accumulate the relative transforms: the global pose of frame $i$ is $M_i = M_{i-1} T_i$.

ICP is described in the [Registration](registration.ipynb) section.

In [ ]:
# estimate the sensor pose of every frame.
poses = sequential_registration(frames, iteration=50)

# compare the estimated trajectory with the ground truth.
# the ground-truth poses are expressed relative to the first frame,
# because SLAM uses the first frame as the origin.
gt_relative_poses = [
    tm.composite([np.linalg.inv(gt_poses[0]), pose]) for pose in gt_poses
]
estimated_trajectory = trajectory(poses)
gt_trajectory = trajectory(gt_relative_poses)

print("estimated trajectory (sensor positions):")
print(estimated_trajectory)
print("max position error:", np.abs(estimated_trajectory - gt_trajectory).max())

The estimated sensor positions match the ground truth, so the frame-to-frame registration successfully recovered the trajectory.

## Mapping: merge the frames into a global map
Once we know the pose of every frame, we can transform each frame back into the world coordinate and merge them into a single map.

In [ ]:
print(inspect.getsource(build_map))

In [ ]:
# build the global map and the trajectory line.
global_map = build_map(frames, poses)
print("number of map points:", len(global_map))

# trajectory as a line connecting consecutive sensor positions.
trajectory_lines = np.stack(
    [estimated_trajectory[:-1], estimated_trajectory[1:]], axis=1
)  # (num_frames - 1, 2, 3)

# visualize the merged map (blue) and the trajectory (red).
obj_map = jv.point(global_map, single_color("#0000ff", len(global_map)))
obj_traj = jv.line(trajectory_lines, single_color("#ff0000", len(trajectory_lines)), width=0.01)
obj_traj_points = jv.point(estimated_trajectory, single_color("#ff0000", len(estimated_trajectory)), point_size=0.02)
jv.display([obj_map, obj_traj, obj_traj_points])

Because the poses were estimated accurately, every frame collapses back onto the same scene, so the merged map reproduces the original point cloud instead of a blurred overlap of misaligned frames. The red line shows the estimated sensor trajectory.

## Summary
- Point cloud SLAM estimates the sensor trajectory and builds a map at the same time.
- The front-end (odometry) estimates frame-to-frame motion by registration (ICP) and accumulates it into a global trajectory.
- Mapping merges the frames using the estimated poses.

A real SLAM additionally has to deal with **sensor noise**, **occlusion** (each frame observes only a part of the scene), and **drift** (small errors accumulate over a long trajectory). The back-end reduces drift with **loop closure** and **pose-graph optimization**. See [docs/slam.md](../docs/slam.md) for related packages such as PythonRobotics.